# Geodesica in spaziotempo di Kerr — Coordinate Kerr-Schild
Unità geometriche $G = c = 1$, lunghezze e tempi in unità di $M$.

## 0. Imports

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

---
## 1. Derivative Ladder

Dato il vettore di stato $(x,y,z)$ calcoliamo in ordine:

| Layer | Output |
|-------|--------|
| 1 | $r$, $\partial_i r$ |
| 2 | $S = r^4 + a^2z^2$, $\partial_i S$ |
| 3 | $f = 2Mr^3/S$, $\partial_i f$ |
| 4 | $l^\mu$, $\partial_i l^\mu$ |
| 5 | $\partial_k g_{\mu\nu}$ |
| 6 | $\bar{F}^i$, $G$, $F^i$ |

In [ ]:
def compute_r(x, y, z, a):
    """
    Layer 1: coordinata radiale KS e sue derivate spaziali.

    In KS, r non è sqrt(x²+y²+z²). È definita implicitamente da:
        (x²+y²)/(r²+a²) + z²/r² = 1
    che si risolve esplicitamente come:
        K = (x²+y²+z²-a²)/2
        W = sqrt(K²+a²z²)
        r² = K + W

    Per a=0: r = sqrt(x²+y²+z²), W = K = r²/2 → tutto torna.

    Le derivate seguono dalla regola della catena:
        ∂_i r² = ∂_i K + ∂_i W
        → ∂_i r = (∂_i K + ∂_i W) / (2r)
    """
    K  = 0.5 * (x**2 + y**2 + z**2 - a**2)
    W  = np.sqrt(K**2 + a**2 * z**2)
    r2 = K + W
    r  = np.sqrt(r2)

    # derivate di K
    dKx, dKy, dKz = x, y, z

    # derivate di W = sqrt(K²+a²z²)
    dWx = K * x / W
    dWy = K * y / W
    dWz = z * (K + a**2) / W

    # derivate di r
    drx = (dKx + dWx) / (2.0 * r)
    dry = (dKy + dWy) / (2.0 * r)
    drz = (dKz + dWz) / (2.0 * r)

    return r, W, np.array([drx, dry, drz])


def compute_f_and_l(x, y, z, r, W, dr, M, a):
    """
    Layer 2-4: scalare f, vettore nullo l^μ e le loro derivate.

    f = 2Mr³/S,  S = r⁴+a²z² = 2r²W
      → f = Mr/W   (forma semplificata, utile per le derivate)

    l^μ = (-1, lx, ly, lz)  con
        lx = (rx+ay)/ρ²,  ly = (ry-ax)/ρ²,  lz = z/r,  ρ²=r²+a²

    Nota: l_μ (covariante, con indice abbassato) = (1, lx, ly, lz)
    perché η^{tt}=-1.
    """
    rho2 = r**2 + a**2   # ρ² = r²+a²

    # --- f e ∂f ---
    # f = M*r/W
    f = M * r / W
    # ∂_i f = (M/W)(∂_i r - (r/W) ∂_i W)
    # ∂_i W già noto da Layer 1 (ricostruiamo qui per chiarezza)
    K   = 0.5*(x**2+y**2+z**2-a**2)
    dWx = K*x/W;  dWy = K*y/W;  dWz = z*(K+a**2)/W
    dW  = np.array([dWx, dWy, dWz])
    df  = (M/W) * (dr - (r/W)*dW)   # shape (3,)

    # --- l^μ ---
    lx_s =  (r*x + a*y) / rho2   # componente spaziale x
    ly_s =  (r*y - a*x) / rho2   # componente spaziale y
    lz_s =   z / r                # componente spaziale z
    # l^μ contravariante = (-1, lx, ly, lz)
    # l_μ covariante     = ( 1, lx, ly, lz)  [alzare/abbassare solo t cambia segno]
    l_up  = np.array([-1.0, lx_s, ly_s, lz_s])   # contravariante
    l_cov = np.array([ 1.0, lx_s, ly_s, lz_s])   # covariante

    # --- ∂_i l^μ  (solo componenti spaziali; l^t=-1=cost) ---
    # Fattori di forma (dal summary, eqs. 8-11)
    Lx_shape = (x*(a**2 - r**2) - 2*a*r*y) / rho2**2
    Ly_shape = (y*(a**2 - r**2) + 2*a*r*x) / rho2**2
    Lz_shape = -z / r**2

    # ∂_i lx = (r δ_{ix} + a δ_{iy})/ρ² + Lx ∂_i r
    dlx = np.array([
        r/rho2 + Lx_shape*dr[0],   # ∂_x lx
        a/rho2 + Lx_shape*dr[1],   # ∂_y lx
              + Lx_shape*dr[2]     # ∂_z lx
    ])
    # ∂_i ly = (r δ_{iy} - a δ_{ix})/ρ² + Ly ∂_i r
    dly = np.array([
        -a/rho2 + Ly_shape*dr[0],  # ∂_x ly
         r/rho2 + Ly_shape*dr[1],  # ∂_y ly
                + Ly_shape*dr[2]   # ∂_z ly
    ])
    # ∂_i lz = δ_{iz}/r + Lz ∂_i r
    dlz = np.array([
               Lz_shape*dr[0],     # ∂_x lz
               Lz_shape*dr[1],     # ∂_y lz
        1.0/r+ Lz_shape*dr[2]      # ∂_z lz
    ])
    # dl[k, mu] = ∂_k l_mu  (solo mu=1,2,3 cioè x,y,z; mu=0 è costante)
    dl_spatial = np.array([dlx, dly, dlz])   # shape (3,3): dl[i,j]=∂_i l_{j+1}

    return f, df, l_up, l_cov, dl_spatial


def compute_dg(f, df, l_cov, dl_spatial):
    """
    Layer 5: derivate spaziali della metrica.

    g_μν = η_μν + f l_μ l_ν
    ∂_k g_μν = (∂_k f) l_μ l_ν + f (∂_k l_μ) l_ν + f l_μ (∂_k l_ν)

    Restituisce dg[k, mu, nu] = ∂_k g_{μν}  per k,μ,ν ∈ {0,1,2,3}.
    Indici: 0=t, 1=x, 2=y, 3=z.
    """
    # l covariante completo (4 componenti)
    l4 = l_cov                          # shape (4,)

    # ∂_k l_μ completo: la componente temporale è costante → derivata = 0
    # dl4[k, mu]: k=0,1,2 → derivata rispetto a x,y,z; mu=0,1,2,3
    dl4 = np.zeros((3, 4))
    dl4[:, 1:] = dl_spatial             # le tre componenti spaziali

    # prodotto esterno l_μ l_ν
    ll = np.outer(l4, l4)               # shape (4,4)

    dg = np.zeros((3, 4, 4))            # dg[k, mu, nu]
    for k in range(3):
        # (∂_k f) l_μ l_ν
        term1 = df[k] * ll
        # f (∂_k l_μ) l_ν
        term2 = f * np.outer(dl4[k], l4)
        # f l_μ (∂_k l_ν)
        term3 = f * np.outer(l4, dl4[k])
        dg[k] = term1 + term2 + term3

    return dg


def compute_acceleration(x, y, z, vx, vy, vz, M, a):
    """
    Layer 6: accelerazione geodetica F^i.

    Dai simboli di Christoffel splittati:
        Γ^μ_αβ = Γ̄^μ_αβ - (f/2) l^μ G_αβ

    L'accelerazione diventa (eq. 13 del summary):
        dv^i/dt = F̄^i + (f/2) G (l^i + v^i)

    con:
        F̄^i = -Γ̄^i_αβ v^α v^β + Γ̄^t_αβ v^α v^β v^i
        G   = G_αβ v^α v^β
        G_αβ = l^σ(∂_α g_{βσ} + ∂_β g_{ασ} - ∂_σ g_{αβ})

    Qui v^α = (1, vx, vy, vz) (cioè v^t = dt/dt = 1).
    """
    # --- Ladder layers 1-5 ---
    r, W, dr = compute_r(x, y, z, a)
    f, df, l_up, l_cov, dl_sp = compute_f_and_l(x, y, z, r, W, dr, M, a)
    dg = compute_dg(f, df, l_cov, dl_sp)  # dg[k,mu,nu], k=0,1,2 → x,y,z

    # 4-velocità coordinata: v^μ = (1, vx, vy, vz)
    v4 = np.array([1.0, vx, vy, vz])

    # --- Simboli di Christoffel "piatti" Γ̄^μ_αβ ---
    # Γ̄^μ_αβ = (1/2) η^{μσ}(∂_α η_{βσ} + ∂_β η_{ασ} - ∂_σ η_{αβ})
    #         + (f/2) l^μ ... ma la parte "piatta" usa solo la metrica piatta.
    # In realtà la split è:
    #   Γ^μ_αβ(g) = Γ̄^μ_αβ(η) + δΓ^μ_αβ
    # dove Γ̄ sono i Christoffel calcolati con g_μν al posto di η_μν
    # ma con derivate di g (non di η). Lo split utile è:
    #   Γ̄^μ_αβ = (1/2) g^{μσ}(∂_α g_{βσ} + ∂_β g_{ασ} - ∂_σ g_{αβ})
    #            tranne che per il termine con l^μ.
    #
    # Seguiamo la formulazione del summary:
    #   F^i = F̄^i + (f/2) G (l^i + v^i)
    # con F̄^i calcolato dai Christoffel "full" privati del termine f·l^μ G_αβ.
    #
    # Christoffel completi:
    #   Γ^μ_αβ = (1/2) g^{μσ}(∂_α g_{βσ} + ∂_β g_{ασ} - ∂_σ g_{αβ})

    # Inversa della metrica: g^{μν} = η^{μν} - f l^μ l^ν
    eta_inv = np.diag([-1.0, 1.0, 1.0, 1.0])
    g_inv = eta_inv - f * np.outer(l_up, l_up)

    # dg ha solo le derivate spaziali (k=x,y,z → indici 1,2,3).
    # Costruiamo dg_full[sigma, mu, nu] con sigma=0,1,2,3:
    # sigma=0 (t): metrica stazionaria → ∂_t g_{μν} = 0
    dg_full = np.zeros((4, 4, 4))
    dg_full[1:, :, :] = dg              # sigma=1,2,3 ← dg[0,1,2]

    # Christoffel completi: Γ^μ_αβ = (1/2) g^{μσ}(∂_α g_{βσ} + ∂_β g_{ασ} - ∂_σ g_{αβ})
    # B[α,β,σ] = ∂_α g_{βσ} + ∂_β g_{ασ} - ∂_σ g_{αβ}
    # dg_full[σ,μ,ν] = ∂_σ g_{μν}, con σ=0 → ∂_t = 0 (metrica stazionaria)
    B = (dg_full.transpose(0,1,2)   # ∂_α g_{βσ}  → B[α,β,σ]
       + dg_full.transpose(1,0,2)   # ∂_β g_{ασ}
       - dg_full.transpose(1,2,0))  # -∂_σ g_{αβ}  NB: NON transpose(2,0,1)!
    # Gamma[μ,α,β] = (1/2) g^{μσ} B[α,β,σ]
    Gamma = 0.5 * np.einsum('ms,abs->mab', g_inv, B)

    # --- F̄^i e F̄^t tramite Christoffel completi ---
    # a_mu = -Γ^μ_αβ v^α v^β
    accel_full = -np.einsum('mab,a,b->m', Gamma, v4, v4)  # shape (4,)

    # Correzione per il fatto che t non è parametro affine:
    #   dv^i/dt = accel_full^i - accel_full^t * v^i
    #           = F̄^i (include già il termine f·G·(l^i+v^i) se usiamo g completa)
    Fi = accel_full[1:] - accel_full[0] * np.array([vx, vy, vz])

    return Fi

print("Derivative ladder definita.")

---
## 2. Sistema ODE

$$\frac{d}{dt}\begin{pmatrix}\mathbf{x}\\ \mathbf{v}\end{pmatrix} = \begin{pmatrix}\mathbf{v}\\ F^i(\mathbf{x},\mathbf{v})\end{pmatrix}$$

In [ ]:
def geodesic_rhs(t, Y, M, a):
    """
    Membro destro del sistema ODE 6D.
    Y = [x, y, z, vx, vy, vz]
    """
    x, y, z, vx, vy, vz = Y
    Fi = compute_acceleration(x, y, z, vx, vy, vz, M, a)
    return [vx, vy, vz, Fi[0], Fi[1], Fi[2]]

print("ODE rhs definita.")

---
## 3. Quantità conservate

Il vettore di stato non contiene $\dot{t} = dt/d\lambda$. Lo recuperiamo dalla definizione di $E$:
$$\dot{t}(t) = \frac{E_0}{1 - f(1 + l_i v^i)}$$

Poi usiamo $\dot{t}$ per calcolare $N$, $E$, $L_z$.

In [ ]:
def compute_tdot_and_conserved(x, y, z, vx, vy, vz, M, a, E0=None):
    """
    Calcola tdot, N, E, Lz dato lo stato (x,y,z,vx,vy,vz).

    Se E0 è None, calcola tdot dalla normalizzazione (usato per le IC).
    Se E0 è fornita, recupera tdot invertendo l'espressione di E.
    """
    r, W, dr = compute_r(x, y, z, a)
    f, df, l_up, l_cov, dl_sp = compute_f_and_l(x, y, z, r, W, dr, M, a)

    # l_i v^i  (somma sugli indici spaziali)
    livi = l_cov[1]*vx + l_cov[2]*vy + l_cov[3]*vz

    # Denominatore comune: 1 - f(1 + l_i v^i)
    denom = 1.0 - f*(1.0 + livi)

    if E0 is None:
        # Calcolo tdot dalla normalizzazione g_{μν} u^μ u^ν = -1
        # N+1 = tdot² [ -(1-f) + 2f l_i v^i (-1) + v² + f(l_i v^i)² ] + 1 = 0
        # → tdot = 1/sqrt(|g_{μν}(1,v^i)(1,v^j)|)
        v2  = vx**2 + vy**2 + vz**2
        bracket = (-(1-f) - 2*f*livi + v2 + f*livi**2)
        # g_{μν} u^μ u^ν / tdot² = bracket  → tdot² = -1/bracket (deve essere >0)
        tdot = 1.0 / np.sqrt(-bracket)
    else:
        # Recupera tdot dalla conservazione di E
        tdot = E0 / denom

    # --- Energia E = tdot * (1 - f(1 + l_i v^i)) ---
    E = tdot * denom

    # --- Momento angolare assiale Lz ---
    # Lz = tdot [ x vy - y vx - (af(x²+y²))/(r²+a²) * (1 + l_i v^i) ]
    Lz = tdot * (x*vy - y*vx - a*f*(x**2+y**2)/(r**2+a**2) * (1.0+livi))

    # --- Normalizzazione N = g_{μν} u^μ u^ν + 1 ---
    v2  = vx**2 + vy**2 + vz**2
    N   = tdot**2 * (-(1-f) - 2*f*livi + v2 + f*livi**2) + 1.0

    return tdot, E, Lz, N


print("Conserved quantities definite.")

---
## 4. Test 1.3 — Orbita circolare Schwarzschild

Parametri: $M=1$, $a=0$, $p=50M$, $e=0$.
Stato iniziale: $Y_0 = (50, 0, 0, 0, 0.14142136, 0)$.

Per un'orbita circolare in Schwarzschild a $r=50M$:
$$v_\phi = \sqrt{M/r^3} \cdot r = \sqrt{M/r} \approx \sqrt{1/50} \approx 0.14142$$
che è esattamente il valore dato.

In [ ]:
# --- Parametri ---
M = 1.0
a = 0.0

# Condizioni iniziali
Y0 = np.array([50.0, 0.0, 0.0, 0.0, 0.14142136, 0.0])

# Periodo orbitale: T = 2π r^{3/2} / sqrt(M) = 2π * 50^{3/2} ≈ 2220
T_orb = 2*np.pi * 50.0**1.5 / np.sqrt(M)
print(f"Periodo orbitale stimato: T ≈ {T_orb:.2f} M")

# Verifica condizioni iniziali
x0, y0, z0, vx0, vy0, vz0 = Y0
tdot0, E0, Lz0, N0 = compute_tdot_and_conserved(x0, y0, z0, vx0, vy0, vz0, M, a)
print(f"Verifica IC:")
print(f"  E0  = {E0:.10f}")
print(f"  Lz0 = {Lz0:.10f}")
print(f"  N0  = {N0:.2e}   (deve essere |N0| ≲ 1e-12)")
print(f"  tdot0 = {tdot0:.10f}")

In [ ]:
# --- Integrazione ---
n_orbits = 5
t_span   = (0.0, n_orbits * T_orb)
t_eval   = np.linspace(0, n_orbits * T_orb, 100_000)

sol = solve_ivp(
    geodesic_rhs,
    t_span,
    Y0,
    method='DOP853',
    t_eval=t_eval,
    rtol=1e-11,
    atol=1e-13,
    args=(M, a)
)

print(f"Integrazione: {sol.message}")
print(f"Passi eseguiti: {sol.t.size}")

In [ ]:
# --- Post-processing: calcolo quantità conservate lungo la traiettoria ---
xs, ys, zs   = sol.y[0], sol.y[1], sol.y[2]
vxs, vys, vzs = sol.y[3], sol.y[4], sol.y[5]

N_arr  = np.zeros(len(sol.t))
E_arr  = np.zeros(len(sol.t))
Lz_arr = np.zeros(len(sol.t))

for i in range(len(sol.t)):
    _, E_i, Lz_i, N_i = compute_tdot_and_conserved(
        xs[i], ys[i], zs[i], vxs[i], vys[i], vzs[i], M, a, E0=E0
    )
    N_arr[i]  = N_i
    E_arr[i]  = E_i
    Lz_arr[i] = Lz_i

dN  = np.abs(N_arr  - N_arr[0])
dE  = np.abs(E_arr  - E0) / np.abs(E0)
dLz = np.abs(Lz_arr - Lz0) / np.abs(Lz0)

print(f"Max |ΔN|  = {dN.max():.2e}")
print(f"Max δE    = {dE.max():.2e}")
print(f"Max δLz   = {dLz.max():.2e}")

In [ ]:
# --- Grafici ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Orbita nel piano x-y
ax = axes[0]
ax.plot(xs, ys, lw=0.8, color='royalblue')
ax.set_aspect('equal')
ax.set_xlabel('x / M')
ax.set_ylabel('y / M')
ax.set_title('Orbita circolare Schwarzschild (5 periodi)')
# cerchio di riferimento
theta_ref = np.linspace(0, 2*np.pi, 500)
ax.plot(50*np.cos(theta_ref), 50*np.sin(theta_ref), 'r--', lw=1, label='r = 50 M (atteso)')
ax.legend()

# Deriva delle quantità conservate
ax2 = axes[1]
t_plot = sol.t / T_orb   # in unità di periodi
ax2.semilogy(t_plot, dN,  label=r'$|N(t)-N(0)|$',          color='steelblue')
ax2.semilogy(t_plot, dE,  label=r'$|E(t)-E_0|/E_0$',       color='tomato')
ax2.semilogy(t_plot, dLz, label=r'$|L_z(t)-L_{z,0}|/L_{z,0}$', color='seagreen')
ax2.set_xlabel('t / T_orb')
ax2.set_ylabel('Deriva relativa')
ax2.set_title('Conservazione delle quantità di moto')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('orbit_schwarzschild_circular.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salvata.")